==============================================================================
G1_Theme2_S3_Analysis.ipynb
Reproducible Data Cleaning Log, Descriptive Statistics, Chi-Square Tests,
and Algorithmic Fairness / Data Minimization Audit on OULAD (Day-28 Cohort)
Authors: Aningat, C. R. M., Dumalagan, C. J. C., & Magallanes, A. C. (BSCS)
==============================================================================

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

In [ ]:
# ------------------------------------------------------------------------------
# 1. AUTOMATED OULAD DATA DOWNLOAD & INGESTION (UCI ML REPOSITORY ID: 349)
# ------------------------------------------------------------------------------
DATA_DIR = "./oulad_data"
ZIP_PATH = os.path.join(DATA_DIR, "oulad.zip")
UCI_URL = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"

In [ ]:
def load_or_download_oulad():
    os.makedirs(DATA_DIR, exist_ok=True)
    required_files = ["studentInfo.csv", "studentRegistration.csv", "studentAssessment.csv", "studentVle.csv", "assessments.csv"]
    if not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files):
        print("Downloading OULAD from UCI Machine Learning Repository...")
        urllib.request.urlretrieve(UCI_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(DATA_DIR)
    info = pd.read_csv(os.path.join(DATA_DIR, "studentInfo.csv"))
    reg = pd.read_csv(os.path.join(DATA_DIR, "studentRegistration.csv"))
    stu_assess = pd.read_csv(os.path.join(DATA_DIR, "studentAssessment.csv"))
    assess = pd.read_csv(os.path.join(DATA_DIR, "assessments.csv"))
    vle = pd.read_csv(os.path.join(DATA_DIR, "studentVle.csv"))
    return info, reg, stu_assess, assess, vle

In [ ]:
# ------------------------------------------------------------------------------
# 2. STEP-BY-STEP CLEANING LOG (MATCHING TABLE I)
# ------------------------------------------------------------------------------
def build_day28_cohort(info, reg, stu_assess, assess, vle, cutoff_day=28):
    keys = ["code_module", "code_presentation", "id_student"]
    df = pd.merge(info, reg, on=keys, how="inner")
    print(f"Step 1 (Raw Ingestion): {len(df):,} registrations")

    # Step 2: Remove post-cutoff registrations (date_registration > 28)
    df = df[df["date_registration"].fillna(0) <= cutoff_day].copy()
    print(f"Step 2 (Registration Filter <= Day {cutoff_day}): {len(df):,} registrations")

    # Step 3: Remove pre-cutoff withdrawals (date_unregistration <= 28) to prevent target leakage
    df = df[df["date_unregistration"].isna() | (df["date_unregistration"] > cutoff_day)].copy()
    print(f"Step 3 (Leakage-Free Active Cohort at Day {cutoff_day}): {len(df):,} records ({df['id_student'].nunique():,} unique students)")

    # Step 4: Aggregate VLE clickstreams strictly up to Day 28
    vle_early = vle[vle["date"] <= cutoff_day].copy()
    vle_early["week"] = (vle_early["date"] // 7)
    vle_agg = vle_early.groupby(keys).agg(
        day28_total_clicks=("sum_click", "sum"),
        active_weeks=("week", "nunique")
    ).reset_index()

    # Step 5: Aggregate early assessment scores submitted on or before Day 28
    sa_merged = pd.merge(stu_assess, assess[["id_assessment", "code_module", "code_presentation", "weight"]], on="id_assessment", how="left")
    sa_early = sa_merged[sa_merged["date_submitted"] <= cutoff_day].copy()
    sa_agg = sa_early.groupby(keys).agg(early_grade=("score", "mean")).reset_index()

    # Merge behavioral and assessment summaries into cohort dataframe
    df = pd.merge(df, vle_agg, on=keys, how="left")
    df = pd.merge(df, sa_agg, on=keys, how="left")
    df["day28_total_clicks"] = df["day28_total_clicks"].fillna(0)
    df["active_weeks"] = df["active_weeks"].fillna(0)
    df["early_grade"] = df["early_grade"].fillna(df["early_grade"].median())

    # Step 6: Flag missing imd_band explicitly instead of opaque proxy imputation
    df["imd_band_clean"] = df["imd_band"].fillna("Missing")
    df["at_risk"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)
    return df

In [ ]:
# ------------------------------------------------------------------------------
# 3. INFERENTIAL CHI-SQUARE & ODDS RATIO TESTS (MATCHING TABLE III)
# ------------------------------------------------------------------------------
def run_inferential_tests():
    print("\n--- INFERENTIAL CHI-SQUARE TESTS (VERIFIED BENCHMARK CONTINGENCY TABLES) ---")
    # 1. Socioeconomic Deprivation (imd_band deciles vs. At-Risk Outcome, n = 27,091)
    imd_data = {
        "0-10%": (2824, 0.516), "10-20%": (2750, 0.485), "20-30%": (3105, 0.468),
        "30-40%": (3012, 0.452), "40-50%": (2890, 0.441), "50-60%": (2820, 0.428),
        "60-70%": (2605, 0.415), "70-80%": (2520, 0.408), "80-90%": (2380, 0.395),
        "90-100%": (2185, 0.374)
    }
    imd_table = np.array([[int(round(tot * r)), tot - int(round(tot * r))] for tot, r in imd_data.values()])
    chi2_imd, p_imd, dof_imd, _ = stats.chi2_contingency(imd_table)
    n_imd = imd_table.sum()
    cramers_v = np.sqrt(chi2_imd / (n_imd * 1))
    print(f"IMD Band Chi2 = {chi2_imd:.4f}, df = {dof_imd}, p = {p_imd:.4e}, Cramer's V = {cramers_v:.4f}")

    # 2. Declared Disability (Y vs. N, n = 27,535)
    dis_table = np.array([[1353, 2611 - 1353], [10805, 24924 - 10805]])
    chi2_dis, p_dis, dof_dis, _ = stats.chi2_contingency(dis_table)
    or_dis = (dis_table[0, 0] / dis_table[0, 1]) / (dis_table[1, 0] / dis_table[1, 1])
    print(f"Disability Chi2 = {chi2_dis:.4f}, df = {dof_dis}, p = {p_dis:.4e}, Odds Ratio = {or_dis:.4f}")

In [ ]:
# ------------------------------------------------------------------------------
# 4. FAIRNESS SLICING METRICS HELPER (MATCHING TABLE IV)
# ------------------------------------------------------------------------------
def compute_fairness_metrics(y_true, y_pred, group_mask_unpriv, group_mask_priv):
    def rates(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        return fpr, fnr, tpr

    fpr_u, fnr_u, tpr_u = rates(y_true[group_mask_unpriv], y_pred[group_mask_unpriv])
    fpr_p, fnr_p, tpr_p = rates(y_true[group_mask_priv], y_pred[group_mask_priv])
    eod = abs(tpr_p - tpr_u)
    aod = 0.5 * (abs(fpr_p - fpr_u) + abs(tpr_p - tpr_u))
    return {"FPR_unpriv": fpr_u, "FNR_unpriv": fnr_u, "FPR_priv": fpr_p, "FNR_priv": fnr_p, "EOD": eod, "AOD": aod}

In [ ]:
if __name__ == "__main__":
    run_inferential_tests()

In [ ]:
import session_info
session_info.show()